In [0]:

# COMMAND ----------
# 1. Definir nombres de catalogo, esquemas y volumen
CATALOGO = "proyecto_final"
ESQUEMAS = ["landing", "bronze", "silver", "gold"]
VOLUMEN = "raw_data"
NOMBRE_PROYECTO = "ventas_retail_william_barboza"

# COMMAND ----------
# 2. Crear Catalogo si no existe
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")

# Comprobacion de existencia del Catalogo extrayendo el valor por posicion de columna
catalogos_existentes = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]

if CATALOGO in catalogos_existentes:
    print(f"OK: Catalogo '{CATALOGO}' verificado/creado con exito.")
else:
    raise Exception(f"ERROR: El catalogo '{CATALOGO}' no se pudo crear.")

# COMMAND ----------
# 3. Crear Esquemas si no existen
for esquema in ESQUEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{esquema}")

# Comprobacion de existencia de los Esquemas extrayendo el valor por posicion de columna
esquemas_existentes = [row[0] for row in spark.sql(f"SHOW SCHEMAS IN {CATALOGO}").collect()]

for esquema in ESQUEMAS:
    if esquema in esquemas_existentes:
        print(f"OK: Esquema '{CATALOGO}.{esquema}' verificado/creado.")
    else:
        print(f"ADVERTENCIA: El esquema '{esquema}' no figura en la lista.")

# COMMAND ----------
# 4. Crear Volumen en la capa Landing
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.landing.{VOLUMEN}")

# Comprobacion de existencia del Volumen usando dbutils
volume_path = f"/Volumes/{CATALOGO}/landing/{VOLUMEN}"

try:
    dbutils.fs.ls(volume_path)
    print(f"OK: Volumen verificado en la ruta: {volume_path}")
except Exception as e:
    print(f"ERROR: El volumen no existe o no se puede acceder a la ruta: {volume_path}")

# COMMAND ----------
# 5. Comprobar/Crear la estructura de carpetas de las entidades con dbutils
proyecto_path = f"{volume_path}/{NOMBRE_PROYECTO}"
entidades = ["clientes", "productos", "pedidos", "detalle_pedidos"]

for entidad in entidades:
    entidad_path = f"{proyecto_path}/{entidad}"
    try:
        dbutils.fs.mkdirs(entidad_path)
        print(f"OK: Directorio listo en: {entidad_path}")
    except Exception as e:
        print(f"ERROR al crear la carpeta {entidad_path}: {e}")

# COMMAND ----------
# 6. Listar contenido actual de la carpeta del proyecto
try:
    archivos = dbutils.fs.ls(proyecto_path)
    print(f"Contenido actual de la carpeta del proyecto ({len(archivos)} elementos):")
    for archivo in archivos:
        print(f" - {archivo.name}")
except Exception as e:
    print(f"ADVERTENCIA: No se pudo listar el contenido: {e}")